In [5]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
gemini_client = genai.Client()

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [2]:
from rag_helper import RAGBase
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [12]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=gemini_client,
    instructions=instructions,
    model="gemini-2.5-flash"
)

In [14]:
assistant.rag("How do I run Ollama locally?")

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [11]:
assistant.rag("How do I run Olama locally?")

'Based on the context provided, here is how you can install and run Ollama locally:\n\n### Step 1: Install Ollama\nFirst, visit [https://ollama.com/download](https://ollama.com/download), choose your operating system, and follow the instructions:\n* **macOS**: Download the `.pkg` and install it.\n* **Windows**: Download the `.msi` and install it.\n* **Linux**: Run the following command in your terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n### Step 2: Run the Model Locally\nOnce installed, open your terminal and run the following command to download and start the LLaMA 3 model:\n```bash\nollama run llama3\n```\nThis command will:\n* Download the LLaMA 3 model (~4GB).\n* Start the model locally.\n* Open a chat-like interface where you can type your questions.\n\n### Step 3: Test the Local Server\nTo verify that your Ollama local server is running, execute this command in another terminal window:\n```bash\ncurl http://localhost:11434\n```\nYou should rece

In [26]:
from google.genai import types
from google import genai

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"]
    }
}

tools = types.Tool(function_declarations=[search_tool])
config = types.GenerateContentConfig(tools=[tools])

messages = ["I just discovered the course. Can I join it?"]

answer = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=messages,
    config=config
)

In [27]:
answer.candidates

[Candidate(
   content=Content(
     parts=[
       Part(
         function_call=FunctionCall(
           args={
             'query': 'join course'
           },
           name='search'
         ),
         thought_signature=b'\n\xf4\x03\x01\x0c9\xd6\xc7\xf4\x9a\xe7\x94\x1a\xc5\x1c\x0f?\xa9\xe7\xb0\xe5\xf8\x7f\xe3\xf3\xd0\xe3\xb6k\x9a\xa9b\xab\x06kx\xd0\xe2Dm\xef\x18\x10CtT\xf7\xd1T\xb0a\xe4\xc2\xca\xaf\x8d\x13Z\xfa:\xf8BK\xc3\x0b\xde\x1d["\xdf\x16\x02\xf4\xb2\x95\x12N<(\xe1\xc5\xaf2\x87\xe6\x8a\xc1\xde=\xadm^\xf8\x84\x07\x87\xdc...'
       ),
     ],
     role='model'
   ),
   finish_reason=<FinishReason.STOP: 'STOP'>,
   index=0
 )]

In [28]:
parts = answer.candidates[0].content.parts[0]
fc = parts.function_call
print(f"Model requested function: {fc.name} with args {fc.args}")

Model requested function: search with args {'query': 'join course'}


In [29]:
import json

args = fc.args
results = search(**args)
result_json = json.dumps(results, indent=2)

In [37]:
messages.append(answer.candidates[0].content)

# Response Search
fn_response_part = types.Part.from_function_response(
        name=fc.name,
        response= {"results": result_json}
    )

messages.append(types.Content(role="user", parts=[fn_response_part]))

In [38]:
final_response = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=messages,
        config=config,
    )

print("Final Response:", final_response.text)

Final Response: Yes, you can still join the course! 

However, if you want to receive a certificate, you must submit your project while project submissions are still being accepted. 

You can start learning and working through the materials immediately. Here is how you can get started:
* **No formal registration confirmation is needed:** You can just start learning and submitting homework.
* **Materials:** All materials, videos, and notebooks are available in the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).
* **Course Platform:** You can track deadlines and submit homework on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).
